## Calculate marker gene scores

Load library

In [1]:
# Path-related libraries
import os
from pyhere import here  # For reproducible relative paths
import sys # system specific parameters
from pathlib import Path # File system paths

# AnnData and single-cell analysis libraries
import scanpy as sc       # For preprocessing and visualization of single-cell data
import anndata as ad      # For handling AnnData objects

# Numerical operations
import numpy as np        # For numerical computations and array manipulations
import pandas as pd

# Data visualization
import seaborn as sns        # Statistical data visualization
import matplotlib.pyplot as plt  # Plotting interface
import matplotlib            # Base matplotlib functionality
from matplotlib.backends.backend_pdf import PdfPages  # Save plots to multi-page PDFs

# Parallize processes
from joblib import Parallel, delayed


# Download from internet
import requests # http

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities
import misc as misc   

Parameters

In [2]:
# create directory for marker genes
ma.create_directories(here('data/marker_database'))
# save paths
base_dir = str(here('data/integrate/second_pass/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
objects_dir = os.path.join(base_dir, 'objects') 
markers_dir = Path(here('data/marker_database'))
anndata_dir = str(here('data/anndata/'))

/work/islet_cartography_scrna/data/marker_database Directory already exists!


In [3]:
# Plotting
plt.style.use('default') 

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
 # 
SMALL_SIZE = 4
MEDIUM_SIZE = 6
BIGGER_SIZE = 8
 
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=SMALL_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

sc.set_figure_params(figsize=(2, 2), frameon=False, dpi_save= 300)

sc.settings.figdir = plot_dir
%config InlineBackend.print_figure_kwargs={'facecolor': 'w'}
%config InlineBackend.figure_format='retina'

Pancreatic cells marker genes lists

In [4]:
# Make directory
data_dir = markers_dir

# Define files and URLs (acess: 2025-10-14)
files = {
    "panglaodb_markers.tsv.gz": "https://panglaodb.se/markers/PanglaoDB_markers_27_Mar_2020.tsv.gz",
    "cellmarker20.xlsx": "http://www.bio-bigdata.center/CellMarker_download_files/file/Cell_marker_Human.xlsx"
}

for filename, url in files.items():
    filepath = data_dir / filename
    if not filepath.exists():
        print(f"Downloading {filename}...")
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, stream=True)
        r.raise_for_status()
        with open(filepath, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    else:
        print(f"{filename} already exists, skipping download.")


panglaodb_markers.tsv.gz already exists, skipping download.
cellmarker20.xlsx already exists, skipping download.


In [5]:
# Load files
panglao_df = pd.read_csv(here('data/marker_database/panglaodb_markers.tsv.gz'), sep='\t', compression='gzip')
#cellmarker_df = pd.read_excel(here('data/marker_database/cellmarker20.xlsx'), sheet_name='human')

Make marker genes directories for each database

In [6]:
# Azimuth marker genes https://azimuth.hubmapconsortium.org/references/#Human%20-%20Pancreas (acess: 2025-10-14) 
# manually written as there is no download
azimuth_genes = {
    "cycling": ["UBE2C","TOP2A","CDK1","BIRC5","PBK","CDKN3","MKI67","CDC20","CCNB2","CDCA3"],
    "immune": ["ACP5","APOE","HLA-DRA","TYROBP","LAPTM5","SDS","FCER1G","C1QC","C1QB","SRGN"],
    "quiescent_stellate": ["RGS5","C11orf96","FABP4","CSRP2","IL24","ADIRF","NDUFA4L2","GPX3","IGFBP4","ESAM"],
    "endothelial": ["PLVAP","RGCC","ENG","PECAM1","ESM1","SERPINE1","CLDN5","STC1","MMP1","GNG11"],
    "schwann": ["NGFR","CDH19","UCN2","SOX10","S100A1","PLP1","TSPAN11","WNT16","SOX2","TFAP2A"],
    "activated_stellate": ["COL1A1","COL1A2","COL6A3","COL3A1","TIMP3","TIMP1","CTHRC1","SFRP2","BGN","LUM"],
    "epsilon": ["BHMT","VSTM2L","PHGR1","TM4SF5","ANXA13","ASGR1","DEFB1","GHRL","COL22A1","OLFML3"],
    "gamma": ["PPY","AQP3","MEIS2","ID2","GPC5-AS1","CARTPT","PRSS23","ETV1","PPY2","TUBB2A"],
    "delta": ["SST","RBP4","SERPINA1","RGS2","PCSK1","SEC11C","HHEX","LEPR","MDK","LY6H"],
    "ductal": ["SPP1","MMP7","IGFBP7","KRT7","ANXA4","SERPINA1","LCN2","CFTR","KRT19","SERPING1"],
    "acinar": ["REG1A","PRSS1","CTRB2","CTRB1","REG1B","CELA3A","PRSS2","REG3A","CPA1","CLPS"],
    "beta": ["IAPP","INS","DLK1","INS-IGF2","G6PC2","HADH","ADCYAP1","GSN","NPTX2","C12orf75"],
    "alpha": ["GCG","TTR","PPP1R1A","CRYBA2","TM4SF4","MAFB","GC","GPX3","PCSK2","PEMT"]
}

In [7]:
# panglao map to make cell type names identical to azimuth names
pangolo_to_azimuth_map = {
    "Acinar cells": "acinar",
    "Alpha cells": "alpha",
    "Beta cells": "beta",
    "Delta cells": "delta",
    "Gamma (PP) cells": "gamma",
    "Ductal cells": "ductal",
    "Endothelial cells": "endothelial",
    "Pancreatic stellate cells": "stellate",
    "Peri-islet Schwann cells": "schwann"
}

# Filter marker genes database
panglao_df_flt = panglao_df[
    (panglao_df["organ"].isin(['Pancreas', 'Immune system', 'Vasculature'])) &
    (panglao_df['species'].isin(['Hs', 'Mm Hs'])) & # human specific marker genes
    (panglao_df['sensitivity_human'] > 0.5) & # How frequent this gene is expressed in that particular cell type
    (panglao_df['specificity_human'] < 0.1) # How frequent this gene is expressed in another cell type than the target cell type
].copy()

# Rename cell type labels
panglao_df_flt = panglao_df_flt.copy()
panglao_df_flt['cell_type_simple'] = panglao_df_flt['cell type'].apply(
    lambda x: pangolo_to_azimuth_map.get(x, misc.to_snake_case(x))
)

# Make into dictionary
panglao_genes = panglao_df_flt.groupby('cell_type_simple')['official gene symbol'].apply(list).to_dict()

Save genelists

In [8]:
# Azimuth
rows = []
for celltype, genes in azimuth_genes.items():
    for gene in genes:
        rows.append({"cell_type": celltype, "gene": gene})

df = pd.DataFrame(rows)
df.to_csv(os.path.join(markers_dir, 'azimuth_genes.csv'), index=False)

# Panglao
rows = []
for celltype, genes in panglao_genes.items():
    for gene in genes:
        rows.append({"cell_type": celltype, "gene": gene})

df = pd.DataFrame(rows)
df.to_csv(os.path.join(markers_dir, 'panglao_genes.csv'), index=False)

In [ ]:
# cellmarker2.0 - stalling this for now because their naming is not clear
# cellmarker_df_flt = cellmarker_df[
#     (cellmarker_df['tissue_type'].isin(['Pancreas', 'Pancreatic acinar tissue', 'Pancreatic duct'])) &
#     (cellmarker_df['cancer_type'] == 'Normal') &
#     (cellmarker_df['cell_type'] == 'Normal cell')].copy()

# cellmarker_df_flt = cellmarker_df_flt.dropna(subset=['cell_name', 'Symbol'])

Import data

In [9]:
# AnnData object
adata = ad.read_h5ad(os.path.join(anndata_dir, 'AD_combined.h5ad'))

# Marker genes
azimuth_genes = pd.read_csv(os.path.join(markers_dir, 'azimuth_genes.csv'))
azimuth_genes = azimuth_genes.groupby("cell_type")["gene"].apply(list).to_dict()
panglao_genes = pd.read_csv(os.path.join(markers_dir, 'panglao_genes.csv'))
panglao_genes = panglao_genes.groupby("cell_type")["gene"].apply(list).to_dict()

# Make directory of directies of genes 
marker_genes_dict = {'azimuth' : azimuth_genes, 'panglao' : panglao_genes}

In [16]:
# Combine both databases into one iterable
tasks = []
for database, genesets in marker_genes_dict.items():
    for celltype, genelist in genesets.items():
        score_name = f"{database}_{celltype}_score"
        tasks.append((score_name, genelist))

# Define function to calculate score
def score_one(task):
    score_name, genelist = task
    tmp = sc.tl.score_genes(
        adata=adata,
        gene_list=genelist,
        copy=True,
        score_name=score_name,
        use_raw=False,
    )
    return tmp.obs[[score_name]]

# run in parallel 
score_df = Parallel(n_jobs=3)(
    delayed(score_one)(task) for task in tasks
)

# combine results
score_df = pd.concat(score_df, axis=1)

# ensure same index order as adata
score_df = score_df.loc[adata.obs_names]

# Save file
score_df.to_csv(os.path.join(files_dir, 'marker_gene_scores.csv'))

In [17]:
# markers = pd.read_csv(os.path.join(files_dir, 'marker_gene_scores.csv'), index_col= 0)